# P8 · Recommender System — MovieLens 100K

## Real lesson: RMSE hides popularity bias; evaluate coverage + Precision@K

A recommender optimised for RMSE tends to recommend only popular items, failing
the long tail. This notebook compares a popularity baseline vs. user-based
collaborative filtering vs. SVD matrix factorisation, and evaluates all three
on RMSE, coverage, and Precision@10.

**The punchline:** popularity has the best RMSE but worst coverage. SVD balances
accuracy and coverage. Always evaluate recommenders beyond RMSE.

## Setup

Import libraries and set global constants.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from pathlib import Path
from collections import defaultdict
import urllib.request, zipfile, io

OUT = Path(__file__).parent
rng = np.random.default_rng(42)

## Load Movielens 100K ───────────────────────────────────────────────────────

In [ ]:
def load_movielens():
    """Download and parse MovieLens 100K (ua.base / ua.test split)."""
    url = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
    print(f"Downloading MovieLens 100K from {url}…")
    try:
        req = urllib.request.urlopen(url, timeout=30)
        zf = zipfile.ZipFile(io.BytesIO(req.read()))
        train = pd.read_csv(
            io.StringIO(zf.read("ml-100k/ua.base").decode()),
            sep="\t", names=["user","item","rating","timestamp"]
        )
        test = pd.read_csv(
            io.StringIO(zf.read("ml-100k/ua.test").decode()),
            sep="\t", names=["user","item","rating","timestamp"]
        )
        return train, test
    except Exception as e:
        print(f"Download failed ({e}) — using synthetic ratings.")
        return None, None

def make_synthetic():
    """Reproducible synthetic ratings: 500 users, 200 items."""
    n_users, n_items = 500, 200
    # Latent factors
    U = rng.normal(0, 1, (n_users, 10))
    V = rng.normal(0, 1, (n_items, 10))
    true_ratings = U @ V.T  # (n_users, n_items)
    # Sample ~100k ratings from top-rated (item, user) pairs
    rows, cols, vals = [], [], []
    for u in range(n_users):
        # Each user rates ~20 items
        top_items = np.argsort(true_ratings[u])[-30:]
        sampled = rng.choice(top_items, size=20, replace=False)
        for i in sampled:
            raw = true_ratings[u, i]
            r = int(np.clip(round(1 + (raw - raw.min()) / (raw.max() - raw.min() + 1e-9) * 4), 1, 5))
            rows.append(u + 1); cols.append(i + 1); vals.append(r)
    df = pd.DataFrame({"user": rows, "item": cols, "rating": vals})
    # 80/20 split per user
    train_rows, test_rows = [], []
    for uid, grp in df.groupby("user"):
        grp_s = grp.sample(frac=1, random_state=42)
        cut = max(1, int(len(grp_s) * 0.8))
        train_rows.append(grp_s.iloc[:cut])
        test_rows.append(grp_s.iloc[cut:])
    return pd.concat(train_rows), pd.concat(test_rows)

train_df, test_df = load_movielens()
if train_df is None:
    train_df, test_df = make_synthetic()

print(f"Train: {len(train_df):,} ratings | Test: {len(test_df):,} ratings")
print(f"Users: {train_df['user'].nunique()} | Items: {train_df['item'].nunique()}")

## Utility: Precision@K And Coverage ─────────────────────────────────────────

In [ ]:
def precision_at_k(recommendations, test_df, k=10, threshold=4.0):
    """Precision@K: fraction of top-K recs that are relevant (rating >= threshold)."""
    relevant = defaultdict(set)
    for row in test_df.itertuples():
        if row.rating >= threshold:
            relevant[row.user].add(row.item)
    precisions = []
    for uid, rec_items in recommendations.items():
        if uid not in relevant:
            continue
        hits = len(set(rec_items[:k]) & relevant[uid])
        precisions.append(hits / k)
    return np.mean(precisions) if precisions else 0.0

def catalog_coverage(recommendations, n_items):
    """Fraction of catalog ever recommended."""
    all_recs = set()
    for recs in recommendations.values():
        all_recs.update(recs)
    return len(all_recs) / n_items

## Build User-Item Matrix ─────────────────────────────────────────────────────

In [ ]:
all_items = sorted(set(train_df["item"]) | set(test_df["item"]))
all_users = sorted(train_df["user"].unique())
n_items   = len(all_items)
item_idx  = {item: i for i, item in enumerate(all_items)}
user_idx  = {user: i for i, user in enumerate(all_users)}

R = np.zeros((len(all_users), n_items))
for row in train_df.itertuples():
    if row.user in user_idx and row.item in item_idx:
        R[user_idx[row.user], item_idx[row.item]] = row.rating

# Global mean and per-user means for baseline
global_mean = R[R > 0].mean()
user_means  = np.where(R.sum(axis=1, keepdims=True) > 0,
                        R.sum(axis=1, keepdims=True) / np.maximum((R > 0).sum(axis=1, keepdims=True), 1),
                        global_mean)

## Model 1: Popularity Baseline ────────────────────────────────────────────

In [ ]:
item_popularity = (R > 0).sum(axis=0)  # number of ratings per item
top_popular     = np.argsort(item_popularity)[::-1][:50].tolist()
popular_items   = [all_items[i] for i in top_popular]

pop_recommendations = {}
for uid in all_users:
    rated = set(train_df[train_df["user"] == uid]["item"])
    recs  = [item for item in popular_items if item not in rated][:10]
    pop_recommendations[uid] = recs

# RMSE for popularity baseline (predict global mean)
test_preds_pop = []
test_actuals   = []
for row in test_df.itertuples():
    if row.user in user_idx and row.item in item_idx:
        test_preds_pop.append(global_mean)
        test_actuals.append(row.rating)
rmse_popular = np.sqrt(np.mean((np.array(test_preds_pop) - np.array(test_actuals))**2))

## Model 2: User-Based Collaborative Filtering ────────────────────────────

In [ ]:
# Cosine similarity between users
from sklearn.metrics.pairwise import cosine_similarity
sim_matrix = cosine_similarity(R)

def ubcf_predict(uid_idx, item_idx_val, sim_matrix, R, k=20):
    sims   = sim_matrix[uid_idx].copy()
    sims[uid_idx] = 0  # exclude self
    # Only users who rated this item
    rated_mask = R[:, item_idx_val] > 0
    if rated_mask.sum() == 0:
        return global_mean
    sims_filtered = sims * rated_mask
    top_k = np.argsort(sims_filtered)[::-1][:k]
    top_k = [i for i in top_k if sims_filtered[i] > 0]
    if not top_k:
        return global_mean
    weights = sims_filtered[top_k]
    ratings = R[top_k, item_idx_val]
    return float(np.dot(weights, ratings) / (weights.sum() + 1e-9))

test_preds_ubcf = []
for row in test_df.itertuples():
    if row.user in user_idx and row.item in item_idx:
        pred = ubcf_predict(user_idx[row.user], item_idx[row.item], sim_matrix, R)
        test_preds_ubcf.append(np.clip(pred, 1, 5))
    else:
        test_preds_ubcf.append(global_mean)
rmse_ubcf = np.sqrt(np.mean((np.array(test_preds_ubcf) - np.array(test_actuals))**2))

ubcf_recommendations = {}
for uid in all_users[:200]:  # limit for speed
    ui = user_idx[uid]
    rated = set(train_df[train_df["user"] == uid]["item"])
    scores = {}
    for item in all_items:
        if item not in rated:
            ii = item_idx[item]
            scores[item] = ubcf_predict(ui, ii, sim_matrix, R)
    ubcf_recommendations[uid] = sorted(scores, key=scores.get, reverse=True)[:10]

## Model 3: Svd Matrix Factorization ────────────────────────────────────────

In [ ]:
from numpy.linalg import svd

# Mean-centre the rating matrix
R_centred = R - user_means
R_centred[R == 0] = 0  # don't subtract mean where unrated

# Thin SVD
U_svd, sigma, Vt = svd(R_centred, full_matrices=False)
k_factors = 50
U_k  = U_svd[:, :k_factors]
S_k  = np.diag(sigma[:k_factors])
Vt_k = Vt[:k_factors, :]
R_hat = U_k @ S_k @ Vt_k + user_means  # add back means
R_hat = np.clip(R_hat, 1, 5)

test_preds_svd = []
for row in test_df.itertuples():
    if row.user in user_idx and row.item in item_idx:
        test_preds_svd.append(R_hat[user_idx[row.user], item_idx[row.item]])
    else:
        test_preds_svd.append(global_mean)
rmse_svd = np.sqrt(np.mean((np.array(test_preds_svd) - np.array(test_actuals))**2))

svd_recommendations = {}
for uid in all_users[:200]:
    ui = user_idx[uid]
    rated_items = set(train_df[train_df["user"] == uid]["item"])
    scores = {item: R_hat[ui, item_idx[item]]
              for item in all_items if item not in rated_items}
    svd_recommendations[uid] = sorted(scores, key=scores.get, reverse=True)[:10]

## Precision@10 And Coverage ─────────────────────────────────────────────────

In [ ]:
users_with_recs = set(pop_recommendations) & set(ubcf_recommendations) & set(svd_recommendations)
test_filtered   = test_df[test_df["user"].isin(users_with_recs)]

p10_pop  = precision_at_k(pop_recommendations,  test_filtered)
p10_ubcf = precision_at_k(ubcf_recommendations, test_filtered)
p10_svd  = precision_at_k(svd_recommendations,  test_filtered)

cov_pop  = catalog_coverage(pop_recommendations,  n_items)
cov_ubcf = catalog_coverage(ubcf_recommendations, n_items)
cov_svd  = catalog_coverage(svd_recommendations,  n_items)

models = ["Popularity", "User-CF", "SVD"]
rmses  = [rmse_popular, rmse_ubcf, rmse_svd]
p10s   = [p10_pop, p10_ubcf, p10_svd]
covs   = [cov_pop, cov_ubcf, cov_svd]

summary = pd.DataFrame({"RMSE": rmses, "Precision@10": p10s, "Coverage": covs}, index=models)
print("\n" + summary.round(3).to_string())

## Write Metrics.Md ─────────────────────────────────────────────────────────

In [ ]:
with open(OUT / "metrics.md", "w") as f:
    f.write("# P8 · Recommender System — Metrics\n\n")
    f.write("**Dataset:** MovieLens 100K (or synthetic if download unavailable)\n\n")
    f.write(summary.round(3).to_markdown())
    f.write("\n\n## Key insight\n\n")
    f.write("The popularity baseline may have competitive RMSE but terrible coverage — "
            "it recommends the same blockbusters to everyone.\n")
    f.write("SVD improves coverage and precision simultaneously.\n")
    f.write("**Never evaluate a recommender system on RMSE alone.**\n")

## Plots ─────────────────────────────────────────────────────────────────────

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
colors = ["#3498db", "#e74c3c", "#2ecc71"]
for ax, col, title in zip(axes, ["RMSE", "Precision@10", "Coverage"],
                           ["RMSE (lower = better)", "Precision@10 (higher = better)",
                            "Catalog coverage (higher = better)"]):
    bars = ax.bar(models, summary[col], color=colors)
    ax.set_title(title)
    ax.set_ylim(0, summary[col].max() * 1.3)
    for bar, val in zip(bars, summary[col]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{val:.3f}", ha="center", va="bottom", fontsize=9)
plt.suptitle("Recommender comparison: RMSE alone misleads", fontsize=11)
plt.tight_layout()
plt.savefig(OUT / "rmse_comparison.png", dpi=120)
plt.close()

print("\nOutputs written: metrics.md, rmse_comparison.png")
print("\nLesson: popularity baseline has narrow coverage — recommends blockbusters only.")
print("SVD discovers long-tail items while maintaining accuracy.")